In [1]:
import matplotlib.pyplot as plt
import h5py
import hdf5plugin
import matplotlib
import numpy as np
import os
import time
%matplotlib ipympl

In [2]:
root = '/data/visitors/danmax/20251117/2025110808/'
raw = os.path.join(root , 'raw')
sample = os.path.join(raw , 'al1050_15pct_center_slice')

In [3]:
scans = [int(file.split('-')[1].split('.h5')[0]) for file in os.listdir(sample) if '_pilatus' not in file and file.endswith('.h5')]
scans = np.sort(scans)
scans = scans[scans >= 48 ] # we restarted with better intensity

In [4]:
dty = np.zeros((1000_000,))
omega = np.zeros((len(dty),))
i = 0
for scan in scans:
    file = 'scan-'+str(scan).zfill(4)+'.h5'
    if '_pilatus' not in file and file.endswith('.h5'):
        with h5py.File( os.path.join(sample, file)) as f:
            if 'im_x' in f['entry/instrument'].keys():
                im_x = f['entry/instrument/im_x/value'][:]
                tom_ry = f['entry/instrument/tom_ry/value'][:]
                assert len(im_x)==len(tom_ry), str(len(im_x)) +' != '+ str(len(tom_ry))
                dty[i:i+len(im_x)] = im_x
                omega[i:i+len(tom_ry)] = tom_ry
                print(file, im_x[0], im_x[-1])
                i += len(im_x)
dty = dty[0:i]
omega = omega[0:i]

scan-0048.h5 -0.75 -0.59
scan-0049.h5 -0.61 -0.45
scan-0050.h5 -0.47 -0.31
scan-0051.h5 -0.33 -0.17
scan-0052.h5 -0.19 -0.03
scan-0053.h5 -0.05 0.11
scan-0054.h5 0.09 0.25
scan-0055.h5 0.23 0.39
scan-0056.h5 0.37 0.53
scan-0057.h5 0.51 0.67
scan-0058.h5 0.65 0.75


In [5]:
np.save('dty_al.npy', dty)
np.save('omega_al.npy', omega)

In [6]:
def collect_sum_int_per_frame(frames):
    s = np.zeros( (frames.shape[0], ) )
    chunk = 250
    i = 0
    while i < frames.shape[0] - chunk - 1:
        s[i:i+chunk]  = frames[i:i+chunk].clip(0).sum(axis=(-2, -1))
        i += chunk
        print(end=str(i)+', ')
    s[i:frames.shape[0]] = frames[i:frames.shape[0]].clip(0).sum(axis=(-2, -1))
    return s

from concurrent.futures import ThreadPoolExecutor

def process_scan(scan):
    file = f'scan-{str(scan).zfill(4)}.h5'
    if '_pilatus' in file or not file.endswith('.h5'):
        return None
    with h5py.File(os.path.join(sample, file)) as f:
        if 'im_x' in f['entry/instrument'].keys():
            frames = f['entry/instrument/pilatus/data']
            s = collect_sum_int_per_frame(frames)
            print(file, frames.shape, frames.nbytes/1e9)
            return s
    return None

t1 = time.perf_counter()
with ThreadPoolExecutor() as ex:
    results = list(ex.map(process_scan, scans))
sum_ints = np.concatenate([r for r in results if r is not None])
np.save('sum_ints_al.npy', sum_ints)
t2 = time.perf_counter()
print(t2 - t1)


250, 250, 500, 500, 250, 250, 750, 750, 250, 250, 250, 500, 500, 1000, 500, 500, 250, 250, 500, 750, 750, 1000, 250, 1250, 750, 750, 500, 500, 750, 1000, 250, 1000, 1250, 500, 1500, 1000, 1000, 750, 750, 1000, 1250, 500, 1250, 1500, 750, 1750, 1250, 1250, 1000, 1000, 1250, 1500, 750, 1500, 1750, 1000, 2000, 1500, 1500, 1250, 1250, 1500, 1750, 1000, 1750, 2000, 1250, 2250, 1750, 1750, 1500, 1500, 1750, 2000, 1250, 2000, 2250, 1500, 2500, 2000, 2000, 1750, 1750, 2000, 2250, 1500, 2250, 2500, 1750, 2750, 2250, 2250, 2000, 2000, 2250, 2500, 1750, 2500, 2750, 2000, 3000, 2500, 2500, 2250, 2250, 2500, 2750, 2000, 2750, 3000, 2250, 3250, 2750, 2750, 2500, 2500, 2750, 3000, 2250, 3000, 3250, 2500, 3500, 3000, 3000, 2750, 2750, 3000, 3250, 2500, 3250, 3500, 2750, 3750, 3250, 3250, 3000, 3000, 3250, 3500, 2750, 3500, 3750, 3000, 4000, 3500, 3500, 3250, 3250, 3500, 3750, 3000, 3750, 4000, 3250, 4250, 3750, 3750, 3500, 3500, 3750, 4000, 3250, 4000, 4250, 3500, 4500, 4000, 4000, 3750, 3750, 4000, 4